# Module 09 — Parallel tool calls

**THE ONE IDEA:** one assistant turn may contain **several** tool calls. Return **all**
their results in a **single** message, or the model quietly stops parallelising.

That failure has no error and no warning. The agent just gets slower and more expensive
over time while every individual response still looks correct.

This module has no theory file behind it — it is a gap in `8.agents/`, added because the
behaviour bites in practice.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, time
from _providers import get_client
from _tools import openai_schemas, run_tool

client, MODEL, _ = get_client("openai")

# Three INDEPENDENT lookups. Nothing here needs an earlier answer, so a competent
# model should ask for all three at once.
QUESTION = ("Look up three policies and report them: the LTV rules, the early "
            "repayment charges, and the proof-of-income requirements.")
print("independent subtasks -> the model should batch them")

## The right way — all results in one message

In [ ]:
def run(batch_results=True, max_steps=6, verbose=True):
    messages = [{"role": "user", "content": QUESTION}]
    per_turn = []
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, max_tokens=600,
                                           tools=openai_schemas(["search_policy"]),
                                           messages=messages)
        msg = r.choices[0].message
        if r.choices[0].finish_reason != "tool_calls":
            return msg.content, per_turn
        n = len(msg.tool_calls)
        per_turn.append(n)
        if verbose: print(f"  step {step}: {n} tool call(s) in ONE turn")
        messages.append(msg)

        blocks = [{"role": "tool", "tool_call_id": tc.id,
                   "content": run_tool(tc.function.name, json.loads(tc.function.arguments))}
                  for tc in msg.tool_calls]
        if batch_results:
            messages.extend(blocks)          # all of them, before the next API call
        else:
            messages.extend(blocks[:1])      # THE BUG: only the first result goes back
            for tc in msg.tool_calls[1:]:    # the rest are answered on a LATER turn
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": "(result withheld until next turn)"})
    return None, per_turn

print("RUN A — batching results correctly")
ans_a, turns_a = run(batch_results=True)
print(f"\ncalls per turn: {turns_a}   total turns: {len(turns_a)}")

## The wrong way — starve the model of results

Same question, but the model does not get every result it asked for. It learns, inside
this one conversation, that asking for three is pointless.

In [ ]:
print("RUN B — withholding results")
ans_b, turns_b = run(batch_results=False)
print(f"\ncalls per turn: {turns_b}   total turns: {len(turns_b)}")

## Why it matters — the cost of losing parallelism

In [ ]:
def sequential_cost(turns):
    """Every extra TURN re-sends the whole conversation. Turns are the expensive unit."""
    return len(turns)

print(f"{'run':22} {'calls/turn':>12} {'turns':>7}")
print("-" * 46)
print(f"{'A batched (correct)':22} {str(turns_a):>12} {sequential_cost(turns_a):>7}")
print(f"{'B withheld (bug)':22} {str(turns_b):>12} {sequential_cost(turns_b):>7}")
print()
print("LESSON — a parallel turn is N tool calls for the price of ONE round trip.")
print("Serialise them and you pay N round trips, and each one re-sends the entire")
print("growing conversation (module 02). Latency and cost both scale with TURNS,")
print("not with tool calls.")
print()
print("The dangerous part is the silence. Withholding a result is not an API error.")
print("The model simply adapts, stops batching, and your agent gets slower over")
print("weeks. Nothing in your logs says why.")
print()
print("RULE: every tool_use id the model sent you gets a result in the NEXT message.")
print("A failed tool still gets one - an error string, or is_error:true on Anthropic.")
print("Never drop one. Module 13 turns that rule into a guard.")

---

**Next:** `10_rag_as_a_tool.ipynb`